In [0]:

# Gold Layer: Dimensional Modeling 
# Reads cleaned Silver Delta tables, constructs conformed dimensions and fact tables using deterministic hash surrogate keys, and incrementally merges directly into ADLS Gen2 Gold Delta tables.

import datetime
from decimal import Decimal
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = "adb_saas_dev"
GOLD_BASE = "abfss://gold@datalakesaas.dfs.core.windows.net"

print("Connected to Unity Catalog:")
print(f"Catalog: {CATALOG}")
print(f"Silver schema: {CATALOG}.silver")
print(f"Gold schema:   {CATALOG}.gold")


# 1. Helper Functions for Unity Catalog Delta Persistence

def write_gold_delta(df, table_name, location=None, pk_cols=None, overwrite=False):
    """
    Writes or merges a DataFrame into a Unity Catalog gold table.
    Creates an external table at the ADLS location if it doesn't exist.
    """
    if overwrite or not spark.catalog.tableExists(table_name):
        print(f"Writing (initial create / overwrite) table: {table_name}")
        writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if location:
            writer = writer.option("path", location)
        writer.saveAsTable(table_name)
    else:
        print(f"Merging into table: {table_name}")
        match_pred = " AND ".join([f"target.{col} = source.{col}" for col in pk_cols])
        delta_target = DeltaTable.forName(spark, table_name)
        delta_target.alias("target").merge(
            df.alias("source"),
            match_pred
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
    print(f"✓ Saved: {table_name.split('.')[-1]}")


# 2.Dimensions (dim_date, dim_plan, dim_organization, dim_user)

# A. dim_date (Dynamic 10-Year Rolling Window)

# COMMAND ----------
start_year = 2020
end_year = datetime.datetime.now().year + 10

df_dates = spark.sql(f"""
    SELECT explode(sequence(to_date('{start_year}-01-01'), to_date('{end_year}-12-31'), interval 1 day)) as date_actual
""")

df_dim_date = df_dates \
    .withColumn("date_key", F.date_format("date_actual", "yyyyMMdd").cast("int")) \
    .withColumn("day_of_month", F.dayofmonth("date_actual")) \
    .withColumn("month", F.month("date_actual")) \
    .withColumn("month_name", F.date_format("date_actual", "MMMM")) \
    .withColumn("quarter", F.quarter("date_actual")) \
    .withColumn("year", F.year("date_actual")) \
    .withColumn("is_weekend", F.dayofweek("date_actual").isin([1, 7])) \
    .select("date_key", "date_actual", "day_of_month", "month", "month_name", "quarter", "year", "is_weekend")

# Append default -1 unknown date member for missing/null dates
df_unknown_date = spark.createDataFrame(
    [(-1, datetime.date(1970, 1, 1), 1, 1, "Unknown", 1, 1970, False)],
    schema=df_dim_date.schema
)

df_dim_date_final = df_dim_date.unionByName(df_unknown_date)

write_gold_delta(
    df_dim_date_final,
    f"{CATALOG}.gold.dim_date",
    location=f"{GOLD_BASE}/dim_date",
    pk_cols=["date_key"],
    overwrite=True
)


# B. dim_plan (With Unknown Member -1)

df_cln_plans = spark.table(f"{CATALOG}.silver.cln_plans")

# Deterministic surrogate key based on natural business key
df_dim_plan_active = df_cln_plans \
    .withColumn("plan_key", F.md5(F.col("plan_id").cast("string"))) \
    .select("plan_key", "plan_id", "plan_name", "tier", "monthly_fee")

# Default -1 orphan member
df_unknown_plan = spark.createDataFrame(
    [("-1", -1, "Unknown Plan", "unknown", "0.0")],
    schema=df_dim_plan_active.schema
)

df_dim_plan = df_dim_plan_active.unionByName(df_unknown_plan)

write_gold_delta(
    df_dim_plan,
    f"{CATALOG}.gold.dim_plan",
    location=f"{GOLD_BASE}/dim_plan",
    pk_cols=["plan_id"],
    overwrite=False
)


# C. dim_organization (With Unknown Member -1)

df_cln_orgs = spark.table(f"{CATALOG}.silver.cln_organizations")

# Deterministic surrogate key based on org_id
df_dim_org_active = df_cln_orgs \
    .withColumn("org_key", F.md5(F.col("org_id").cast("string"))) \
    .withColumnRenamed("name", "org_name") \
    .select("org_key", "org_id", "org_name", "industry", "country_code", "is_active")

# Default -1 orphan member
df_unknown_org = spark.createDataFrame(
    [("-1", -1, "Unknown Organization", "Unknown", "XX", False)],
    schema=df_dim_org_active.schema
)

df_dim_org = df_dim_org_active.unionByName(df_unknown_org)

write_gold_delta(
    df_dim_org,
    f"{CATALOG}.gold.dim_organization",
    location=f"{GOLD_BASE}/dim_organization",
    pk_cols=["org_id"],
    overwrite=False
)

# D. dim_user (Resolves org_key via dim_organization, With Unknown Member -1)

df_cln_users = spark.table(f"{CATALOG}.silver.cln_users")

# Look up org_key from dim_organization
dim_org_lookup = spark.table(f"{CATALOG}.gold.dim_organization").select("org_id", "org_key")

df_dim_user_active = df_cln_users.alias("u") \
    .join(dim_org_lookup.alias("o"), F.col("u.org_id") == F.col("o.org_id"), "left") \
    .withColumn("user_key", F.md5(F.col("u.user_id").cast("string"))) \
    .withColumn("org_key", F.coalesce(F.col("o.org_key"), F.lit("-1"))) \
    .select("user_key", "user_id", "org_key", "email", "role", "is_active")

# Default -1 orphan member
df_unknown_user = spark.createDataFrame(
    [("-1", -1, "-1", "unknown@system.local", "Unknown", False)],
    schema=df_dim_user_active.schema
)

df_dim_user = df_dim_user_active.unionByName(df_unknown_user)

write_gold_delta(
    df_dim_user,
    f"{CATALOG}.gold.dim_user",
    location=f"{GOLD_BASE}/dim_user",
    pk_cols=["user_id"],
    overwrite=False
)

# 3. Fact Tables (fact_billing, fact_subscriptions, fact_feature_usage, fact_logins)

# A. fact_billing

df_cln_invoices = spark.table(f"{CATALOG}.silver.cln_invoices")
df_cln_subs = spark.table(f"{CATALOG}.silver.cln_subscriptions")

# Join invoice -> subscription to get org_id and plan_id
df_inv_with_sub = df_cln_invoices.alias("i") \
    .join(df_cln_subs.alias("s"), F.col("i.subscription_id") == F.col("s.subscription_id"), "left") \
    .select(
        F.col("i.invoice_id"),
        F.col("i.billing_date"),
        F.col("i.amount_due"),
        F.col("i.amount_paid"),
        F.col("i.status"),
        F.col("i.is_anomaly"),
        F.col("s.org_id"),
        F.col("s.plan_id")
    )

# Dimension lookups for foreign keys
dim_org_lookup = spark.table(f"{CATALOG}.gold.dim_organization").select("org_id", "org_key")
dim_plan_lookup = spark.table(f"{CATALOG}.gold.dim_plan").select("plan_id", "plan_key")

df_fact_billing = df_inv_with_sub.alias("b") \
    .join(dim_org_lookup.alias("o"), F.col("b.org_id") == F.col("o.org_id"), "left") \
    .join(dim_plan_lookup.alias("p"), F.col("b.plan_id") == F.col("p.plan_id"), "left") \
    .withColumn("billing_key", F.md5(F.col("b.invoice_id").cast("string"))) \
    .withColumn("date_key", F.coalesce(F.date_format("b.billing_date", "yyyyMMdd").cast("int"), F.lit(-1))) \
    .withColumn("org_key", F.coalesce(F.col("o.org_key"), F.lit("-1"))) \
    .withColumn("plan_key", F.coalesce(F.col("p.plan_key"), F.lit("-1"))) \
    .select(
        "billing_key",
        "invoice_id",
        "date_key",
        "org_key",
        "plan_key",
        "amount_due",
        "amount_paid",
        "status",
        "is_anomaly"
    )

write_gold_delta(
    df_fact_billing,
    f"{CATALOG}.gold.fact_billing",
    location=f"{GOLD_BASE}/fact_billing",
    pk_cols=["invoice_id"],
    overwrite=False
)

# B. fact_subscriptions (MRR & Contract State)

dim_org_lookup = spark.table(f"{CATALOG}.gold.dim_organization").select("org_id", "org_key")
dim_plan_lookup = spark.table(f"{CATALOG}.gold.dim_plan").select("plan_id", "plan_key", "monthly_fee")

df_fact_subscriptions = df_cln_subs.alias("s") \
    .join(dim_org_lookup.alias("o"), F.col("s.org_id") == F.col("o.org_id"), "left") \
    .join(dim_plan_lookup.alias("p"), F.col("s.plan_id") == F.col("p.plan_id"), "left") \
    .withColumn("subscription_fact_key", F.md5(F.col("s.subscription_id").cast("string"))) \
    .withColumn("date_key", F.coalesce(F.date_format("s.started_at", "yyyyMMdd").cast("int"), F.lit(-1))) \
    .withColumn("org_key", F.coalesce(F.col("o.org_key"), F.lit("-1"))) \
    .withColumn("plan_key", F.coalesce(F.col("p.plan_key"), F.lit("-1"))) \
    .withColumn("is_active", F.col("s.status") == "active") \
    .withColumn(
        "mrr_amount",
        F.when(F.col("s.status") == "active", F.coalesce(F.col("p.monthly_fee"), F.lit(0.00))).otherwise(F.lit(0.00))
    ) \
    .select(
        "subscription_fact_key",
        "subscription_id",
        "date_key",
        "org_key",
        "plan_key",
        "status",
        "mrr_amount",
        "is_active"
    )

write_gold_delta(
    df_fact_subscriptions,
    f"{CATALOG}.gold.fact_subscriptions",
    location=f"{GOLD_BASE}/fact_subscriptions",
    pk_cols=["subscription_id"],
    overwrite=False
)

# C. fact_feature_usage (Incremental MERGE on Composite Grain)

df_cln_usage = spark.table(f"{CATALOG}.silver.cln_feature_usage")

dim_org_lookup = spark.table(f"{CATALOG}.gold.dim_organization").select("org_id", "org_key")
dim_user_lookup = spark.table(f"{CATALOG}.gold.dim_user").select("user_id", "user_key")

df_fact_feature_usage = df_cln_usage.alias("u") \
    .join(dim_org_lookup.alias("o"), F.col("u.org_id") == F.col("o.org_id"), "left") \
    .join(dim_user_lookup.alias("usr"), F.col("u.user_id") == F.col("usr.user_id"), "left") \
    .withColumn("date_key", F.coalesce(F.date_format("u.usage_date", "yyyyMMdd").cast("int"), F.lit(-1))) \
    .withColumn("org_key", F.coalesce(F.col("o.org_key"), F.lit("-1"))) \
    .withColumn("user_key", F.coalesce(F.col("usr.user_key"), F.lit("-1"))) \
    .withColumn(
        "usage_fact_key",
        F.md5(F.concat_ws("||", F.col("date_key"), F.col("org_key"), F.col("user_key"), F.col("u.feature_name")))
    ) \
    .select(
        "usage_fact_key",
        "date_key",
        "org_key",
        "user_key",
        "feature_name",
        "usage_count"
    )

write_gold_delta(
    df_fact_feature_usage,
    f"{CATALOG}.gold.fact_feature_usage",
    location=f"{GOLD_BASE}/fact_feature_usage",
    pk_cols=["date_key", "org_key", "user_key", "feature_name"],
    overwrite=False
)

# D. fact_logins

df_cln_logins = spark.table(f"{CATALOG}.silver.cln_logins")

dim_user_lookup = spark.table(f"{CATALOG}.gold.dim_user").select("user_id", "user_key", "org_key")

df_fact_logins = df_cln_logins.alias("l") \
    .join(dim_user_lookup.alias("u"), F.col("l.user_id") == F.col("u.user_id"), "left") \
    .withColumn("login_fact_key", F.md5(F.col("l.event_id").cast("string"))) \
    .withColumn("date_key", F.coalesce(F.date_format("l.event_timestamp", "yyyyMMdd").cast("int"), F.lit(-1))) \
    .withColumn("user_key", F.coalesce(F.col("u.user_key"), F.lit("-1"))) \
    .withColumn("org_key", F.coalesce(F.col("u.org_key"), F.lit("-1"))) \
    .select(
        "login_fact_key",
        "event_id",
        "date_key",
        "org_key",
        "user_key",
        F.col("l.event_timestamp").alias("login_timestamp"),
        F.col("l.login_success").alias("is_success"),
        "client_device"
    )

write_gold_delta(
    df_fact_logins,
    f"{CATALOG}.gold.fact_logins",
    location=f"{GOLD_BASE}/fact_logins",
    pk_cols=["event_id"],
    overwrite=False
)
